# Imports

In [35]:
import pandas as pd
import numpy as np

# Data Entry

In [36]:
# reading data
data = pd.read_csv("train.csv")


data["Time_taken(min)"]=  data["Time_taken(min)"].str.strip("(min)").str.strip().astype(float)


# data

# len(data)

# Clean raw timestamps and handle missing/invalid values

In [37]:
# stripping string columns from data
data = data.apply(lambda col: col.str.strip() if col.dtype == "str" else col)

invalids = [
    "nan", "NaN", "NAN", "None", "none", "NONE",
    "null", "Null", "NULL", "NA", "N/A", "n/a",
    "false", "False", "FALSE","unknown", "Unknown"
]



data = data[~ data.isin(invalids).any(axis=1)]

data


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,24.0
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33.0
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26.0
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,21.0
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45588,0x7c09,JAPRES04DEL01,30,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:35:00,11:45:00,conditions Windy,High,1,Meal,motorcycle,0,No,Metropolitian,32.0
45589,0xd641,AGRRES16DEL01,21,4.6,0.000000,0.000000,0.070000,0.070000,16-02-2022,19:55:00,20:10:00,conditions Windy,Jam,0,Buffet,motorcycle,1,No,Metropolitian,36.0
45590,0x4f8d,CHENRES08DEL03,30,4.9,13.022394,80.242439,13.052394,80.272439,11-03-2022,23:50:00,00:05:00,conditions Cloudy,Low,1,Drinks,scooter,0,No,Metropolitian,16.0
45591,0x5eee,COIMBRES11DEL01,20,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:35:00,13:40:00,conditions Cloudy,High,0,Snack,motorcycle,1,No,Metropolitian,26.0


# Remove outliers

In [38]:


n = data["Time_taken(min)"].count()
dt = data["Time_taken(min)"].sort_values()


first_qt = int( n * 0.25 ) + 1
third_qt = int( n * 0.75 ) + 1

q1 = dt.iloc[first_qt]
q3 = dt.iloc[third_qt]

IQR = q3 - q1


upper_fence = q3 + 1.5 * IQR
lower_fence = q1 - 1.5 * IQR



print(f"first_qt: {first_qt} third_qt: {third_qt}")
print(f"Q1 = {q1}, Q3 = {q3}, IQR = {IQR}")
print(f"lower = {lower_fence}, upper = {upper_fence}")
print(f"min: {dt.min()}, max: {dt.max()}")

data = data[( data["Time_taken(min)"] < upper_fence ) &
            ( data["Time_taken(min)"] > lower_fence )
            ]

first_qt: 10343 third_qt: 31027
Q1 = 19.0, Q3 = 33.0, IQR = 14.0
lower = -2.0, upper = 54.0
min: 10.0, max: 54.0


# Calculate Prep Time, Transit Time, Total Delivery Time

In [39]:
time_orderd = data["Time_Orderd"].apply( lambda time: (int(time.split(":")[0])*60)+int(time.split(":")[1]))
time_order_picked = data["Time_Order_picked"].apply( lambda time: (int(time.split(":")[0])*60)+int(time.split(":")[1]))
data["Time_Prep(min)"] = time_order_picked - time_orderd
data = data[data["Time_Prep(min)"] > 0]



data["Transit_Time(min)"] = data["Time_taken(min)"] - data["Time_Prep(min)"]
data = data[data["Transit_Time(min)"] > 0]

data["Total_Delivery_Time(min)"] = data["Transit_Time(min)"] + data["Time_Prep(min)"]


data

# dt = data["Time_Order_picked"] - data["Time_Orderd"]
# dt.astype("str").str.split(" ").str[-1]

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),Time_Prep(min),Transit_Time(min),Total_Delivery_Time(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,...,2,Snack,motorcycle,0,No,Urban,24.0,15,9.0,24.0
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,...,2,Snack,scooter,1,No,Metropolitian,33.0,5,28.0,33.0
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,...,0,Drinks,motorcycle,1,No,Urban,26.0,15,11.0,26.0
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,...,0,Buffet,motorcycle,1,No,Metropolitian,21.0,10,11.0,21.0
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,...,1,Snack,scooter,1,No,Metropolitian,30.0,15,15.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45587,0x1178,RANCHIRES16DEL01,35,4.2,23.371292,85.327872,23.481292,85.437872,08-03-2022,21:45:00,...,2,Drinks,motorcycle,1,No,Metropolitian,33.0,10,23.0,33.0
45588,0x7c09,JAPRES04DEL01,30,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:35:00,...,1,Meal,motorcycle,0,No,Metropolitian,32.0,10,22.0,32.0
45589,0xd641,AGRRES16DEL01,21,4.6,0.000000,0.000000,0.070000,0.070000,16-02-2022,19:55:00,...,0,Buffet,motorcycle,1,No,Metropolitian,36.0,15,21.0,36.0
45591,0x5eee,COIMBRES11DEL01,20,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:35:00,...,0,Snack,motorcycle,1,No,Metropolitian,26.0,5,21.0,26.0


# Create categories for time-of-day and flag delayed orders

In [41]:
def time_of_day(h):
    if 6 <= h < 11:
        return "Breakfast"
    elif 11 <= h < 15:
        return "Lunch Peak"
    elif 15 <= h < 18:
        return "Afternoon"
    elif 18 <= h < 22:
        return "Dinner Rush"
    else:
        return "Late Night"


# Create time-of-day category

time_of_day = data["Time_Orderd"].apply(lambda val : time_of_day(int(val.strip().split(":")[0])) )
data["Time_Of_Day(h)"]= time_of_day

# Create Is_Delayed category

n = data["Total_Delivery_Time(min)"].count()
third_qt = int( n * 0.75 ) + 1
q3 = data["Total_Delivery_Time(min)"].sort_values().iloc[third_qt]
is_delayed = data["Total_Delivery_Time(min)"].apply(lambda val : False if val <= q3 else True)

data["Is_Delayed"] = is_delayed
data


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),Time_Prep(min),Transit_Time(min),Total_Delivery_Time(min),Time_Of_Day(h),Is_Delayed
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,...,motorcycle,0,No,Urban,24.0,15,9.0,24.0,Lunch Peak,False
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,...,scooter,1,No,Metropolitian,33.0,5,28.0,33.0,Dinner Rush,False
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,...,motorcycle,1,No,Urban,26.0,15,11.0,26.0,Breakfast,False
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,...,motorcycle,1,No,Metropolitian,21.0,10,11.0,21.0,Dinner Rush,False
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,...,scooter,1,No,Metropolitian,30.0,15,15.0,30.0,Lunch Peak,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45587,0x1178,RANCHIRES16DEL01,35,4.2,23.371292,85.327872,23.481292,85.437872,08-03-2022,21:45:00,...,motorcycle,1,No,Metropolitian,33.0,10,23.0,33.0,Dinner Rush,False
45588,0x7c09,JAPRES04DEL01,30,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:35:00,...,motorcycle,0,No,Metropolitian,32.0,10,22.0,32.0,Lunch Peak,False
45589,0xd641,AGRRES16DEL01,21,4.6,0.000000,0.000000,0.070000,0.070000,16-02-2022,19:55:00,...,motorcycle,1,No,Metropolitian,36.0,15,21.0,36.0,Dinner Rush,True
45591,0x5eee,COIMBRES11DEL01,20,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:35:00,...,motorcycle,1,No,Metropolitian,26.0,5,21.0,26.0,Lunch Peak,False
